In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import itertools
from itertools import *

import warnings
warnings.filterwarnings('ignore')

In [ ]:
col_names = ['frame', 'interaction_type', 'atom_1', 'atom_2', 'distance']

df = pd.read_csv('contact_data.csv', sep=':' and ',' and '\t', skiprows=[0,1], names=col_names)

def ca_filter(df):
    new_df = df[(df.atom_1.str.contains('2:CA')) & (df.atom_2.str.contains('2:CA'))]
    new_df['res-1'] = new_df['atom_1'].str[2:5]
    new_df['res-2'] = new_df['atom_2'].str[2:5]
    return new_df

df2 = ca_filter(df)

aa_list = ['ARG','ALA','ASN','ASP','CYS','GLU','GLN','GLY','HIS','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
combos = list(itertools.combinations_with_replacement(aa_list, 2))
combos

df3 = df2[['res-1','res-2', 'distance']]

percentages = {key:0 for key in combos}

for i in combos:

    percentages[i] = ((len(df3.where((df3['res-1']==i[0]) & (df3['res-2']==i[1])).dropna()['distance']))
    + (len(df3.where((df3['res-2']==i[0]) & (df3['res-1']==i[1])).dropna()['distance'])))/4500

perc_series = pd.Series(percentages)

array = np.array(combos)
residue1 = array[:,0]
residue2 = array[:,1]
newpercs = np.array(perc_series.to_list())

df4 = pd.DataFrame(array, columns=['res-1', 'res-2'])

df4['%'] = newpercs

array = np.zeros((20,20))
dictionary = {}
for i in range(len(df4)):
    name = (df4.iloc[i]['res-1'],df4.iloc[i]['res-2'])
    dictionary[name] = df4.iloc[i]['%']

for pos1, i in enumerate(aa_list):
    for pos2, j in enumerate(aa_list):
        pair = (i,j)
        if pair in dictionary.keys():
            array[pos1,pos2] = dictionary[pair]
            array[pos2,pos1] = dictionary[pair]
        pair = (j,i)
        if pair in dictionary.keys():
            array[pos1,pos2] = dictionary[pair]
            array[pos2,pos1] = dictionary[pair]

heatmap = sns.heatmap(array,xticklabels=aa_list, yticklabels=aa_list, cmap="YlGnBu")

fig = heatmap.get_figure()

fig.savefig('percentage_heatmap.png')
